# 🧮 다중 팩터 크로스섹셔널 모델 (Multi-Factor Cross-Sectional)

## 이 프로젝트의 다음 장
가격 예측(ML)은 실패했고([docs/FINDINGS.md](../../docs/FINDINGS.md)), 추세추종은 "덜 잃는 법"이었다.
이제 **"무엇을 예측하느냐 + 어떤 정보를 쓰느냐"**를 바꾼다 → **팩터 투자**.

개별 종목의 방향을 맞히려 하지 않고, **매달 유니버스를 여러 팩터로 순위 매겨 상위를 롱**한다.
학술적으로 검증된 정통 접근이며, 분산된 통계적 베팅이라 개별 예측보다 견고하다.

## 팩터 구성
| 팩터 | 정의 | PIT? |
|---|---|---|
| **모멘텀** | 최근 12개월(마지막 1개월 제외) 수익률 | ✅ 가격기반 |
| **저변동성** | 최근 6개월 변동성의 역수 | ✅ 가격기반 |
| **단기반전** | 최근 1개월 수익률 × (-1) | ✅ 가격기반 |
| **밸류** | 이익수익률(1/PER), 1/PBR | ⚠️ 현재값만(실시간 랭킹 전용) |
| **퀄리티** | ROE, 이익률 | ⚠️ 현재값만(실시간 랭킹 전용) |

## ⚠️ 정직한 한계 (반드시 이해)
무료 데이터의 펀더멘털은 **현재 스냅샷**이라 과거 시점 값(point-in-time)이 아니다.
→ 밸류·퀄리티를 **과거 백테스트에 쓰면 look-ahead 편향**. 그래서:
- **백테스트**는 가격기반 3팩터만 (PIT 정확, 정직)
- **실시간 랭킹**에만 펀더멘털을 얹음 (오늘의 추천용, 백테스트 아님)

## 엄격성 (동일 원칙)
- 팩터는 t시점까지 정보로 계산, 다음 리밸런스에 보유 (룩어헤드 방지)
- 거래비용 차감, 월간 리밸런스, 벤치마크 = 유니버스 균등보유 · SPY

> ⚠️ 투자 자문 아님. 교육/연구용.

In [ ]:
# =====================================================================
# ⚙️ Cell 1: 설정 & 유니버스
# =====================================================================
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

# 유니버스: 섹터 분산 대형주 (많을수록 좋으나 yfinance 부하 고려 ~50)
UNIVERSE = [
    'AAPL','MSFT','GOOGL','AMZN','META','NVDA','AVGO','ORCL','CRM','ADBE','AMD','QCOM','TXN','INTC','CSCO',
    'JPM','BAC','WFC','GS','MS','C','V','MA','AXP','BLK',
    'UNH','JNJ','LLY','PFE','ABBV','MRK','TMO','ABT',
    'PG','KO','PEP','WMT','COST','HD','MCD','NKE','SBUX','DIS',
    'XOM','CVX','CAT','BA','HON','GE','UPS','NFLX','T','VZ',
]
BENCH  = 'SPY'
START  = '2010-01-01'
END    = None
COST   = 0.0005     # 편도 거래비용
ANN    = 252
REBAL  = 21         # 월간 리밸런스
TOP_Q  = 0.20       # 상위 20% 롱

print(f'유니버스 {len(UNIVERSE)}종목 | 리밸런스 {REBAL}일 | 상위 {TOP_Q:.0%} 롱 | 비용 {COST*100:.2f}%')

In [ ]:
# =====================================================================
# 📥 Cell 2: 가격 패널 수집 (auto_adjust, 공통 구간 정렬)
# =====================================================================
def fetch_close(tickers):
    frames = {}
    for t in tickers:
        try:
            df = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            s = df['Close'].dropna()
            if len(s) > 300:
                frames[t] = s
        except Exception:
            pass
    return pd.DataFrame(frames)

print('가격 수집 중... (수십 종목이라 1~3분)')
prices = fetch_close(UNIVERSE + [BENCH])
spy = prices[BENCH]
close = prices[[c for c in prices.columns if c != BENCH]].dropna(how='all')
close = close.dropna(axis=1, thresh=int(len(close)*0.8))  # 데이터 부실 종목 제외
close = close.dropna()
spy = spy.reindex(close.index).ffill()
print(f'최종: {close.shape[1]}종목 × {close.shape[0]}일 ({close.index[0].date()}~{close.index[-1].date()})')

In [ ]:
# =====================================================================
# 🧮 Cell 3: 가격기반 팩터 (모두 PIT — 과거 시점 정보만 사용)
# =====================================================================
rets = close.pct_change()

# 모멘텀(12-1): t-252 ~ t-21 수익률 (마지막 1개월 제외 = 단기반전 노이즈 회피)
f_mom = close.shift(21) / close.shift(252) - 1
# 저변동성: 최근 126일 변동성의 '역방향'(변동성 낮을수록 점수↑)
f_lowvol = -(rets.rolling(126).std())
# 단기반전: 최근 21일 수익률 × (-1) (많이 오른 건 되돌림 경향)
f_rev = -(close.pct_change(21))

FACTORS = {'Momentum': f_mom, 'LowVol': f_lowvol, 'Reversal': f_rev}
print('가격기반 팩터 3종 계산 완료:', list(FACTORS.keys()))

In [ ]:
# =====================================================================
# 🧪 Cell 4: 크로스섹셔널 백테스트 엔진 (룩어헤드 방지 + 비용)
# =====================================================================
def zscore_row(row):
    m, s = row.mean(), row.std()
    return (row - m) / s if (s and s > 0) else row * 0.0

def perf(period_returns, rebal=REBAL):
    r = np.asarray(period_returns, float)
    out = {'CAGR': np.nan, 'Vol': 0.0, 'Sharpe': 0.0, 'MDD': 0.0, 'Calmar': np.nan}
    if len(r) == 0:
        return out
    ppy = ANN / rebal
    eq = np.cumprod(1 + r)
    out['CAGR'] = float(eq[-1] ** (ppy / len(r)) - 1) if eq[-1] > 0 else np.nan
    out['Vol'] = float(r.std() * np.sqrt(ppy))
    out['Sharpe'] = float(r.mean() / r.std() * np.sqrt(ppy)) if r.std() > 0 else 0.0
    peak = np.maximum.accumulate(eq)
    out['MDD'] = float(((eq - peak) / peak).min())
    out['Calmar'] = float(out['CAGR'] / abs(out['MDD'])) if out['MDD'] < 0 else np.nan
    return out

def backtest(score_fn, warmup=252):
    # score_fn(date) → 종목별 점수 Series (높을수록 롱 우선)
    dates = close.index
    port, bench, eqcurve_dates = [], [], []
    prev, turnover = set(), 0.0
    i = warmup
    while i < len(dates) - 1:
        d = dates[i]
        score = score_fn(d).dropna()
        if len(score) < 10:
            i += REBAL; continue
        k = max(int(len(score) * TOP_Q), 1)
        picks = list(score.sort_values(ascending=False).head(k).index)
        j = min(i + REBAL, len(dates) - 1)
        d2 = dates[j]
        hr = float((close.loc[d2, picks] / close.loc[d, picks] - 1).mean())
        new = set(picks); n_tr = len(new - prev) + len(prev - new); turnover += n_tr
        cost = (n_tr / max(len(new), 1)) * COST
        port.append(hr - cost)
        bench.append(float((close.loc[d2] / close.loc[d] - 1).mean()))
        eqcurve_dates.append(d2)
        prev = new
        i = j
    return (pd.Series(port, index=eqcurve_dates),
            pd.Series(bench, index=eqcurve_dates), turnover)

def composite_score(date):
    zs = [zscore_row(f.loc[date]) for f in FACTORS.values()]
    return sum(zs) / len(zs)

port_ret, bench_ret, turn = backtest(composite_score)
spy_ret = spy.reindex(port_ret.index).ffill().pct_change().fillna(0)  # 참고용(주기 다름)

m_p, m_b = perf(port_ret.values), perf(bench_ret.values)
print('=== 다중팩터 백테스트 (가격기반 3팩터, 비용반영) ===')
print(f'{"":16}{"CAGR":>9}{"Sharpe":>9}{"MDD":>9}{"Calmar":>9}')
print(f'{"팩터 상위20% 롱":16}{m_p["CAGR"]*100:8.1f}%{m_p["Sharpe"]:9.2f}{m_p["MDD"]*100:8.1f}%{m_p["Calmar"]:9.2f}')
print(f'{"유니버스 균등보유":16}{m_b["CAGR"]*100:8.1f}%{m_b["Sharpe"]:9.2f}{m_b["MDD"]*100:8.1f}%{m_b["Calmar"]:9.2f}')
print(f'\n리밸런스 {len(port_ret)}회 | 연평균 회전 {turn/(len(close)/ANN):.1f}')

In [ ]:
# =====================================================================
# 🔬 Cell 5: 팩터별 기여도 분해 (어떤 팩터가 실제로 먹히나)
# =====================================================================
def single_factor_score(fdf):
    return lambda date: zscore_row(fdf.loc[date])

print('=== 팩터별 단독 성과 (상위20% 롱) ===')
print(f'{"팩터":12}{"CAGR":>9}{"Sharpe":>9}{"MDD":>9}')
rows = {}
for name, fdf in FACTORS.items():
    pr, _, _ = backtest(single_factor_score(fdf))
    mm = perf(pr.values); rows[name] = pr
    print(f'{name:12}{mm["CAGR"]*100:8.1f}%{mm["Sharpe"]:9.2f}{mm["MDD"]*100:8.1f}%')
mc = perf(port_ret.values)
print(f'{"복합(3팩터)":12}{mc["CAGR"]*100:8.1f}%{mc["Sharpe"]:9.2f}{mc["MDD"]*100:8.1f}%')
print('\n해석: 복합 Sharpe가 개별 팩터보다 높으면 → 팩터 분산 효과가 실제로 작동.')

# 자산곡선
plt.figure(figsize=(13, 6))
(1 + port_ret).cumprod().plot(label='복합 팩터 (상위20%)', lw=2.2, color='navy')
(1 + bench_ret).cumprod().plot(label='유니버스 균등보유', lw=1.4, ls='--', color='gray')
for name, pr in rows.items():
    (1 + pr).cumprod().plot(label=f'{name} 단독', lw=1, alpha=0.7)
plt.yscale('log'); plt.title('다중 팩터 크로스섹셔널 — 자산곡선(로그)')
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# =====================================================================
# 🔮 Cell 6: [실시간 랭킹] 가격팩터 + 현재 펀더멘털 결합 → 오늘의 추천
#   ⚠️ 펀더멘털은 '현재 스냅샷'이라 과거 백테스트 불가. 오직 '오늘의 forward 랭킹'용.
# =====================================================================
print('현재 펀더멘털 수집 중... (종목별 조회라 느릴 수 있음)')
fund = {}
for t in close.columns:
    try:
        info = yf.Ticker(t).info
        pe  = info.get('trailingPE')
        pb  = info.get('priceToBook')
        roe = info.get('returnOnEquity')
        pm  = info.get('profitMargins')
        fund[t] = {
            'earnings_yield': (1.0 / pe) if (pe and pe > 0) else np.nan,   # 밸류
            'book_yield':     (1.0 / pb) if (pb and pb > 0) else np.nan,   # 밸류
            'roe':  roe if roe is not None else np.nan,                    # 퀄리티
            'margin': pm if pm is not None else np.nan,                    # 퀄리티
        }
    except Exception:
        fund[t] = {}
fund = pd.DataFrame(fund).T

def z(s):
    s = s.astype(float); m, sd = s.mean(), s.std()
    return (s - m) / sd if (sd and sd > 0) else s * 0

# 밸류/퀄리티 종합 z-score (결측은 0=중립)
value_z   = (z(fund['earnings_yield']) + z(fund['book_yield'])) / 2
quality_z = (z(fund['roe']) + z(fund['margin'])) / 2

# 오늘의 가격팩터 종합
today = close.index[-1]
price_z = composite_score(today)

# 최종 결합 (가격 3팩터 : 밸류 : 퀄리티 = 3 : 1 : 1 가중, 결측 0)
combined = (3 * price_z.reindex(close.columns).fillna(0)
            + value_z.reindex(close.columns).fillna(0)
            + quality_z.reindex(close.columns).fillna(0)) / 5

rank = pd.DataFrame({
    'price_factor': price_z.reindex(close.columns).round(2),
    'value': value_z.reindex(close.columns).round(2),
    'quality': quality_z.reindex(close.columns).round(2),
    'combined': combined.round(3),
}).sort_values('combined', ascending=False)

k = max(int(len(rank) * TOP_Q), 1)
print('\n' + '=' * 64)
print(f' 🔮 오늘의 팩터 랭킹 (기준일 {today.date()}) — 상위 {k}종목 롱 후보')
print('   ⚠️ 펀더멘털 포함 = forward 랭킹(백테스트 아님). 가격팩터만 백테스트됨.')
print('=' * 64)
print(rank.head(k).to_string())
print('=' * 64)
print(' 🏆 롱 후보:', ', '.join(rank.head(k).index.tolist()))

## 🧾 결과 해석 & 정직한 한계

### 백테스트(가격기반 3팩터)에서 볼 것
- **복합 팩터 Sharpe > 균등보유 Sharpe** 이면 → 팩터 순위매기기가 실제로 기여.
- **복합 Sharpe > 개별 팩터 최고값** 이면 → 팩터 분산 효과(서로 다른 시점에 작동)가 진짜.
- 대형주 유니버스라 엣지는 **작을 것**(Sharpe 0.4~0.8이면 성공). 그게 정직한 팩터 수익의 현실.

### 냉정한 한계
1. **펀더멘털은 백테스트 안 됨** — 현재 스냅샷이라 과거 PIT가 없음. Cell 6의 밸류·퀄리티는 **오늘 랭킹에만** 반영, 과거 성과 검증 불가.
2. **생존 편향** — UNIVERSE가 현재 대형주라 과거 백테스트가 낙관적.
3. **대형주는 이미 arbitraged** — 팩터 프리미엄이 소형주·해외보다 약함.

### 진짜로 더 나아가려면 (다음 장)
1. **PIT 펀더멘털 확보** → SEC EDGAR `companyfacts` API(무료, XBRL, 신고일 기준)로 과거 시점 재무를 재구성 → 밸류·퀄리티도 정직하게 백테스트. (데이터 엔지니어링 필요)
2. **유니버스 확대** → S&P500 전체 또는 소형주 포함(엣지↑, 데이터 부담↑).
3. **팩터 개선** → 어닝 리비전(추정치 변화), 어닝 서프라이즈(PEAD) 추가 = "남들이 느리게 반영하는 정보".
4. **롱숏 중립화** → 상위 롱 + 하위 숏으로 시장 베타 제거 → 순수 팩터 알파 측정.

> ⚠️ 파라미터(12-1, 126일, 상위20%)는 학술 관습값. 최적화하면 다시 과적합. 견고성이 가치.